In [1]:
# %load_ext autoreload
# %autoreload 2

# pi-metaboqc: Step-by-Step Pipeline Tutorial

Welcome to the `pi-metaboqc` tutorial. This notebook demonstrates how to execute the comprehensive LC-MS metabolomics data quality control pipeline step by step. 

By running this notebook, you can inspect the intermediate outputs, data structures, and visualization plots generated at each stage of the pipeline.

## Step 0: Environment Setup

First, we will import the core modules of `pi-metaboqc`, configure the input data paths, and create a dedicated output directory for the results of this notebook.

In [2]:
import os
import sys
import json
import pandas as pd
from pathlib import Path
from loguru import logger

# Import core modules of pi-metaboqc
from pimqc import io_utils as iu
from pimqc.dataset_builder import build_dataset
from pimqc.data_quality_assessment import MetaboIntQA
from pimqc.invalid_feature_sample_filtering import MetaboIntFLTR
from pimqc.data_signal_correction import MetaboIntSC
from pimqc.normalization import MetaboIntNorm

# Configure paths (Assuming the notebook is running from the 'examples' directory)
current_dir = Path(os.getcwd())
repo_root = current_dir.parent
data_dir = repo_root / "src" / "pimqc" / "data"

meta_path = data_dir / "project_meta.csv"
int_path = data_dir / "project_intensity.csv"
param_path = data_dir / "pipeline_parameters.json"
# param_path = data_dir / "pipeline_parameters.ymal"

# Create a dedicated folder for the output results of this notebook
output_base_dir = current_dir / "notebook_pipeline_output"
iu._check_dir_exists(dir_path=str(output_base_dir), handle="makedirs")

logger.info(f"Test output directory: {output_base_dir}")

2026-04-07 17:27:12.837 | INFO     | __main__:<module>:30 - Test output directory: c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc\examples\notebook_pipeline_output


## Step 0: Initialization

We need to load the global configuration parameters (`pipeline_parameters.json` or `pipeline_parameters.yaml`), the project metadata (e.g., sample types, batch info), and the raw feature intensity matrix.

In [3]:
# 1. Load pipeline parameters from JSON
pipeline_params = iu._load_json_file(input_file_path=str(param_path))
logger.success("Pipeline parameters loaded successfully.")

# 2. Load metadata and intensity matrix
meta_info = pd.read_csv(filepath_or_buffer=meta_path, header=[0])
int_df = pd.read_csv(filepath_or_buffer=int_path, header=[0],index_col=[0])

logger.info(f"Meta Info Shape: {meta_info.shape}")
logger.info(f"Intensity Matrix Shape: {int_df.shape}")

2026-04-07 17:27:12.844 | SUCCESS  | __main__:<module>:3 - Pipeline parameters loaded successfully.
2026-04-07 17:27:12.875 | INFO     | __main__:<module>:9 - Meta Info Shape: (466, 6)
2026-04-07 17:27:12.876 | INFO     | __main__:<module>:10 - Intensity Matrix Shape: (376, 466)


## Step 1: Build Standardized Dataset

This step merges the metadata and raw intensity matrix into a `MetaboInt` object, which utilizes a MultiIndex structure to securely bind biological groupings, injection orders, and batches to the raw data.

In [4]:
logger.info("▶ Step 1: Building standardized MetaboInt dataset...")
step1_dir = os.path.join(output_base_dir, "01_Raw_Data")

# Extract specific parameters for the MetaboInt initialization
mi_params = pipeline_params.get("MetaboInt", {})

raw_data = build_dataset(
    meta_info=meta_info,
    int_df=int_df,
    pipeline_params=pipeline_params,
    mode=mi_params.get("mode", "POS"),
    batch=mi_params.get("batch", "Batch"),
    sample_type=mi_params.get("sample_type", "Sample Type"),
    bio_group=mi_params.get("bio_group", "Bio Group"),
    sample_name=mi_params.get("sample_name", "Sample Name"),
    inject_order=mi_params.get("inject_order", "Inject Order"),
    output_dir=step1_dir
)

# Check the generated object info
print("Raw Data object type:", type(raw_data))
print("Raw Data Shape:", raw_data.shape)
raw_data.head(3)

2026-04-07 17:27:12.883 | INFO     | __main__:<module>:1 - ▶ Step 1: Building standardized MetaboInt dataset...
2026-04-07 17:27:13.002 | SUCCESS  | pimqc.io_utils:time_wrap:197 - Execution time of "build_dataset": 00:00:00.117.


Raw Data object type: <class 'pimqc.core_classes.MetaboInt'>
Raw Data Shape: (376, 466)


Batch                       B1                                            \
Sample Type              Blank                                             
Bio Group              Unknown                                             
Inject Order               1             2             3             4     
Sample Name           BLANK1-1      BLANK1-2      BLANK1-3      BLANK1-4   
Metabolite                                                                 
1-Methyladenosine     2.621231      5.652807      3.239268      5.257827   
1007                       NaN           NaN           NaN           NaN   
1014               9379.222469  31356.532947  30833.606694  31164.944389   

Batch                                                                          \
Sample Type                   QC         Sample                                 
Bio Group                Unknown        Unknown                                 
Inject Order                 11             12             13             14    
Sample Name                QC1-1         B1-001         B1-004         B1-005   
Metabolite                                                                      
1-Methyladenosine     943.791165    5406.528937    3915.657258    5531.893409   
1007               136449.506322  256945.452939  167877.236064  185285.606438   
1014                87494.289154    7228.023776   15566.477210   47641.572124   

Batch                                            ...             B3  \
Sample Type                                      ...         Sample   
Bio Group                                        ...        Unknown   
Inject Order                 15             16   ...            532   
Sample Name               B1-008         B1-010  ...         B3-379   
Metabolite                                       ...                  
1-Methyladenosine    4943.623517    4185.657935  ...    2568.871521   
1007               352691.763379  336266.762791  ...  295447.672682   
1014                23057.806625   43012.889822  ...   86012.343507   

Batch                                                                      \
Sample Type                                                                 
Bio Group                                                                   
Inject Order                533           534           535           536   
Sample Name              B3-382        B3-388        B3-389        B3-400   
Metabolite                                                                  
1-Methyladenosine   2344.023781   3057.698668   2998.738467   2725.003175   
1007               53826.850181  30100.466332  28752.234790  37889.835098   
1014               81595.293032  18799.980346  26851.713549  31450.529968   

Batch                                                                      \
Sample Type                                                                 
Bio Group                                                                   
Inject Order                537           538           539           540   
Sample Name              B3-405        B3-406        B3-409        B3-439   
Metabolite                                                                  
1-Methyladenosine   3546.294827   3491.341035   3735.114397   2636.204593   
1007               57397.586479  38507.504870  25764.130318  95832.683696   
1014               54888.960507  40806.508264  32111.385257  43721.093153   

Batch                             
Sample Type                   QC  
Bio Group                Unknown  
Inject Order                 541  
Sample Name               QC3-16  
Metabolite                        
1-Methyladenosine    4540.174710  
1007               210709.146918  
1014                58170.629794  

[3 rows x 466 columns]

## Step 3: Raw Data Quality Assessment & Stage-1 Filtering

Before making any modifications, we perform a Quality Assessment (QA) on the raw data (generating PCA, correlation heatmaps, etc.). 
Following the QA, we execute the **Stage-1 Filtering (Missing Value Filtering)** to remove features with excessive missing values based on Global, Group, and QC sample tolerances.

In [5]:
# [Step 2] Quality Assessment on Raw Data
logger.info("▶ Step 2: Conducting QA on Raw Data...")
step2_dir = os.path.join(output_base_dir, "02_QA_Raw_Data")
qa_raw = MetaboIntQA(raw_data, pipeline_params=pipeline_params)
qa_raw.execute_qa(output_dir=step2_dir)

# [Step 3] Stage-1 Missing Value (MV) Filtering
logger.info("▶ Step 3: Filtering invalid features by Missing Value...")
step3_dir = os.path.join(output_base_dir, "03_Filtered_Data")
metabo_fltr = MetaboIntFLTR(raw_data, pipeline_params=pipeline_params)

# Execute the decoupled MV filtering module
filtered_data = metabo_fltr.execute_mv_fltr(output_dir=step3_dir)

print(f"Features before filter: {raw_data.shape[0]}, after MV filter: {filtered_data.shape[0]}")

2026-04-07 17:27:13.023 | INFO     | __main__:<module>:2 - ▶ Step 2: Conducting QA on Raw Data...
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s pruned
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s dropped
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s pruned
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s dropped
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s pruned
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s dropped
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s dropped
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s dropped
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s pruned
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s pruned
[07-04-2026 17:27:13] [fontTools.subset] [INFO] %s pruned
[07-04-2026 17:27:13] [fontTools.subset] [INFO] Added gid0 to subset
[07-04-2026 17:27:13] [fontTools.subset] [INFO] Added first four glyphs to subset
[07-04-2026 17:27:13] [fontTools.subset] [INFO] Closing glyph list over 'GSUB': %d glyphs before
[07-04-2026

Features before filter: 376, after MV filter: 343
